In [26]:
# Instalación condicional de OR-Tools (solo para este notebook)
# Intenta importar; si falla, instala en el entorno actual y reintenta.
import sys, subprocess, importlib, os

def ensure_package(pkg: str, extra_args=None):
    try:
        importlib.import_module(pkg)
        print(f"Paquete '{pkg}' ya disponible.")
        return True
    except ModuleNotFoundError:
        print(f"Paquete '{pkg}' no encontrado. Instalando...")
        args = [sys.executable, "-m", "pip", "install", pkg]
        if extra_args:
            args.extend(extra_args)
        try:
            completed = subprocess.run(args, check=False, capture_output=True, text=True)
            print(completed.stdout)
            if completed.returncode != 0:
                print("Instalación fallida:")
                print(completed.stderr)
                return False
        except Exception as e:
            print(f"Error al invocar pip: {e}")
            return False
        # Reintentar importación
        try:
            importlib.invalidate_caches()
            importlib.import_module(pkg)
            print(f"Paquete '{pkg}' instalado y cargado correctamente.")
            return True
        except Exception as e:
            print(f"Instalación aparente exitosa, pero falla al importar '{pkg}': {e}")
            return False

ok = ensure_package("ortools")
if not ok:
    print("Advertencia: no se pudo instalar 'ortools'. Los bloques MIP usarán modo demostración/fallback.")


Paquete 'ortools' ya disponible.


## Contexto de Negocio

## Empresa y situación
Producción opera con schedule manual: capacidad no balanceada, setup time genera esperas, no hay visibilidad de factibilidad.

## Qué / Por qué / Para qué / Cuándo / Cómo
- **Qué**: Scheduling de producción: asignación de órdenes a máquinas/líneas en horizonte de tiempo minimizando makespan o tardías.
- **Por qué**: Balanceamiento automático, consideración de setup, visibilidad de bottlenecks, evaluación de cambios.
- **Para qué**: Mejora de OEE (eficiencia), reducción de tardías, negociación de lead times con ventas.
- **Cuándo**: Planeamiento semanal o según demanda (job shop vs flow shop).
- **Cómo**: Formulación con constraints, solvers (OR-Tools), heurísticas (SPT, LPT).

# OR-08 · Programación de Producción
 
## 🎯 Objetivos de Aprendizaje
- Entender el problema de programación de producción (capacidad, secuencias, setup).
- Plantear y resolver un modelo básico de scheduling con restricciones.
- Generar artefactos verificables: tablas de secuencia, KPIs de utilización, tiempos de ciclo.
 
## 🧩 Prerrequisitos
- Python 3.10+ y paquetes: `pandas`, `numpy`, `plotly`, `pulp`/`ortools`.
- Datasets: órdenes, rutas, capacidades, tiempos de setup en `data/raw/`.
- (Opcional) Configuración de entorno virtual y kernel de Jupyter.
 
## 🗃️ Datasets
- `orders.csv`: id, sku, qty, due_date, prioridad.
- `products.csv`: sku, familia, tiempo_proceso, tiempo_setup.
- `resources.csv`: máquina, capacidad, calendario.
 
## 🧠 Caso de Uso
- Minimizar retrasos y cambios costosos de setup, maximizando utilización.
- Decisiones: secuencia por máquina, asignación de lotes, cumplimiento de due date.
- KPIs: tardanza promedio, setups realizados, utilización por recurso.
 
## 🧭 Estructura del Notebook
1. Setup y carga de datos
2. Modelado del scheduling (restricciones y objetivo)
3. Solución y validación de resultados
4. Exportación de artefactos (csv/html)
5. Conclusiones
6. Notas de operación (costes, retención, gobernanza)
 
## 🛠️ Notas de Operación (Costes, Retención, Gobernanza)
- Costes: tiempos de cómputo vs tamaño del problema; límites de OR-Tools/PuLP.
- Retención: mantener `raw` 90 días; `analytics` 180 días; `curated` según auditoría.
- Gobernanza: calidad de datos de tiempos de setup y calendarios; reproducibilidad.
 
## 🔗 Referencias
- Pinedo, Scheduling: Theory, Algorithms, and Systems.
- OR-Tools Job Shop, Flow Shop ejemplos.
- Mejoras: modelos híbridos con restricciones suaves/penalizaciones.

# Nota sobre dependencias
Este notebook verifica automáticamente la disponibilidad de `ortools` y realiza una instalación local si falta, únicamente para este entorno de ejecución. Si prefieres gestionar dependencias manualmente, puedes desactivar la celda 1 o instalar `ortools` en tu `venv` previamente:
- Windows PowerShell:
```
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -U pip
pip install ortools
```
- Bash (WSL/Linux/Mac):
```
python -m venv .venv
source .venv/bin/activate
pip install -U pip
pip install ortools
```
La instalación automática no afecta a otros notebooks ni al `requirements.txt` del proyecto.

# Índice rápido
1. Objetivos y Prerrequisitos
2. Configuración del Entorno
3. Carga y Validación de Datos
4. Secuenciación Mono-Máquina (M1) + KPIs + Exportes
5. Modelo PuLP: Objetivo, Restricciones y Solución
6. Validaciones de Capacidad y Almacén
7. Variante Multi-Máquina (M1/M2) + Gantt
8. Comparativa KPIs Mono vs Multi + Interpretación
9. OR-Tools (MIP básico, opcional)
10. Buenas Prácticas de Calendarización
11. Notas de Operación y Referencias

## 1️⃣ Configuración del Entorno

## 🎯 Objetivos de Aprendizaje

- Definir qué aprenderá el lector (máx. 5–7 puntos).
- Conectar con el caso de uso del dominio (demanda, logística, IoT).
- Incluir resultados verificables (métricas, validaciones, artefactos generados).

In [27]:
import pandas as pd
import numpy as np
import pulp
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Rutas
DATA_DIR = Path("../../data/raw")
OUTPUT_DIR = Path("../../data/processed/or08_production_schedule")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Librerías cargadas")
print(f"   PuLP versión: {pulp.__version__}")
print(f"📁 Directorio datos: {DATA_DIR.resolve()}")

✅ Librerías cargadas
   PuLP versión: 3.3.0
📁 Directorio datos: F:\GitHub\supply-chain-data-notebooks\data\raw


In [28]:
# ⚙️ Preparación de entorno y rutas
import sys, platform
from pathlib import Path
import numpy as np, pandas as pd
np.random.seed(42)
 
# Detectar raíz del repo
_candidates = [Path.cwd(), *Path.cwd().parents]
_repo_root = None
for _p in _candidates:
    if (_p / 'pyproject.toml').exists() or (_p / 'src').exists():
        _repo_root = _p
        break
if _repo_root is None:
    _repo_root = Path.cwd()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
print(f"✅ Entorno listo. Raíz del repo: {_repo_root}")
print(f"Python: {platform.python_version()} | pandas: {pd.__version__}")

✅ Entorno listo. Raíz del repo: f:\GitHub\supply-chain-data-notebooks
Python: 3.12.10 | pandas: 2.3.3


## 2️⃣ Generar Datos de Planificación

## Carga de datos
Leer datasets necesarios desde `data/raw/` y validar esquemas esperados (ordenes, productos, recursos).

In [ ]:
# Carga de datos con validación de esquemas
from pathlib import Path
import pandas as pd

DATA_DIR = Path('data/raw')
ORDERS_FILE = DATA_DIR / 'orders.csv'
PRODUCTS_FILE = DATA_DIR / 'products.csv'
RESOURCES_FILE = DATA_DIR / 'resources.csv'

# Leer si existen; si no, crear muestras mínimas
if ORDERS_FILE.exists():
    df_orders = pd.read_csv(ORDERS_FILE, parse_dates=['date','due_date'], dtype={'sku':str})
else:
    df_orders = pd.DataFrame({
        'order_id':[1,2,3],
        'date': pd.to_datetime(['2025-01-01','2025-01-01','2025-01-02']),
        'sku':['SKU-001','SKU-002','SKU-001'],
        'qty':[50,30,40],
        'due_date': pd.to_datetime(['2025-01-03','2025-01-04','2025-01-03']),
        'priority':[2,1,2],
    })

if PRODUCTS_FILE.exists():
    df_products = pd.read_csv(PRODUCTS_FILE, dtype={'sku':str})
else:
    df_products = pd.DataFrame({
        'sku':['SKU-001','SKU-002'],
        'family':['A','B'],
        'proc_time_min':[5,8],
        'setup_time_min':[10,15],
    })

if RESOURCES_FILE.exists():
    df_resources = pd.read_csv(RESOURCES_FILE)
else:
    df_resources = pd.DataFrame({
        'machine':['M1','M2'],
        'capacity_units_per_hour':[600,600],
        'calendar':'08:00-17:00',
    })

print('✅ Datos cargados (o generados):')
for name, df in [('orders', df_orders), ('products', df_products), ('resources', df_resources)]:
    print(f'- {name}: {len(df)} filas | columnas: {list(df.columns)}')

# Validación mínima
def expect_cols(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f'⚠️ {name} faltan columnas: {missing}')
    else:
        print(f'🧪 {name} OK columnas: {cols}')

expect_cols(df_orders, ['date','sku','qty','due_date','priority'], 'orders')
expect_cols(df_products, ['sku','proc_time_min','setup_time_min'], 'products')
expect_cols(df_resources, ['machine','capacity_units_per_hour'], 'resources')

✅ Datos cargados (o generados):
- orders: 60 filas | columnas: ['order_id', 'date', 'sku', 'qty', 'due_date', 'priority']
- products: 10 filas | columnas: ['sku', 'family', 'proc_time_min', 'setup_time_min']
- resources: 2 filas | columnas: ['machine', 'capacity_units_per_hour', 'calendar']
🧪 orders OK columnas: ['date', 'sku', 'qty', 'due_date', 'priority']
🧪 products OK columnas: ['sku', 'proc_time_min', 'setup_time_min']
🧪 resources OK columnas: ['machine', 'capacity_units_per_hour']


## 3️⃣ Visualizar Demanda

## Desarrollo / Modelo
Formular un esquema sencillo de programación (job shop/flow shop) con restricciones de capacidad y tiempos de setup.

In [34]:
# Modelo simple de secuenciación por máquina con penalización por tardanza
import pandas as pd
import numpy as np

# Asegurar tipos correctos
if 'due_date' in df_orders.columns:
    try:
        df_orders['due_date'] = pd.to_datetime(df_orders['due_date'])
    except Exception as e:
        print('⚠️ No se pudo convertir due_date a datetime:', e)

# Construir tareas por orden con tiempos de proceso
jobs = df_orders.merge(df_products[['sku','proc_time_min','setup_time_min']], on='sku', how='left')
jobs['proc_time_min'] = jobs['proc_time_min'].fillna(5)
jobs['setup_time_min'] = jobs['setup_time_min'].fillna(10)

# Asignación heurística: ordenar por due_date y prioridad (SPT/EDD simplificado)
jobs = jobs.sort_values(['due_date','priority','proc_time_min']).reset_index(drop=True)

# Simular secuencia en una máquina (M1) acumulando setups cuando cambia SKU
schedule = []
current_time = pd.Timestamp(jobs['date'].min())
current_sku = None
for _, r in jobs.iterrows():
    setup = r['setup_time_min'] if (current_sku is not None and r['sku'] != current_sku) else 0
    start = current_time + pd.Timedelta(minutes=setup)
    finish = start + pd.Timedelta(minutes=r['proc_time_min']*r['qty'])
    # Calcular tardanza robusto
    due = r['due_date']
    if not isinstance(due, pd.Timestamp):
        try:
            due = pd.to_datetime(due)
        except Exception:
            due = finish  # sin penalización si no se puede parsear
    tardiness = max(pd.Timedelta(0), finish - due)
    schedule.append({
        'order_id': r['order_id'], 'sku': r['sku'], 'start': start, 'finish': finish,
        'setup_min': setup, 'proc_min': r['proc_time_min']*r['qty'], 'tardiness_min': tardiness.total_seconds()/60.0
    })
    current_time = finish
    current_sku = r['sku']

schedule_df = pd.DataFrame(schedule)
print('📅 Secuencia (M1):')
display(schedule_df.head())

# KPIs
kpis = {
    'orders': len(schedule_df),
    'total_setup_min': schedule_df['setup_min'].sum(),
    'avg_tardiness_min': schedule_df['tardiness_min'].mean(),
}
print('📈 KPIs: ', kpis)

📅 Secuencia (M1):


,order_id,sku,start,finish,setup_min,proc_min,tardiness_min
0,7,SKU-010,2025-01-01 00:00:00.000000000,2025-01-01 06:49:59.999999999,0.0,410.0,0.0
1,2,SKU-008,2025-01-01 07:05:17.999999999,2025-01-01 14:17:17.999999999,15.3,432.0,0.0
2,1,SKU-008,2025-01-01 14:17:17.999999999,2025-01-01 16:41:17.999999999,0.0,144.0,0.0
3,16,SKU-010,2025-01-01 16:56:53.999999999,2025-01-01 18:18:53.999999999,15.6,82.0,0.0
4,8,SKU-001,2025-01-01 18:29:53.999999999,2025-01-01 19:35:53.999999999,11.0,66.0,0.0


📈 KPIs:  {'orders': 60, 'total_setup_min': np.float64(581.4999999999999), 'avg_tardiness_min': np.float64(1254.3316666555554)}


## Resultados y hallazgos
Resumir métricas y visualizaciones clave; exportar resultados si corresponde.

In [35]:
# Exportación de artefactos
from pathlib import Path
import json
from typing import Any

OUTPUT_DIR = Path('data/processed/or08')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

schedule_path = OUTPUT_DIR / 'schedule_M1.csv'
schedule_df.to_csv(schedule_path, index=False)
print(f'✅ Exportado: {schedule_path}')

# Serializar KPIs con tipos nativos
def to_native(obj: Any):
    try:
        import numpy as np
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
    except Exception:
        pass
    return obj

kpis_native = {k: to_native(v) for k, v in kpis.items()}

kpis_path = OUTPUT_DIR / 'kpis.json'
with open(kpis_path, 'w', encoding='utf-8') as f:
    json.dump(kpis_native, f, ensure_ascii=False, indent=2)
print(f'✅ Exportado: {kpis_path}')

✅ Exportado: data\processed\or08\schedule_M1.csv
✅ Exportado: data\processed\or08\kpis.json


## 5️⃣ Función Objetivo

## Conclusiones
- La heurística EDD/SPT simplificada genera una secuencia inicial.
- KPIs permiten evaluar setups y tardanza; ajustar reglas o capacidades.
- Próximo: extender a múltiples máquinas y considerar ventanas de calendario.

In [36]:
# Función objetivo PuLP (guardas para ejecución segura)
# Si no existen las variables del modelo, creamos un ejemplo mínimo coherente.
import pulp
import pandas as pd

# Preparar productos y periodos
products = [p for p in (df_products['sku'].unique() if 'sku' in df_products.columns else ['SKU-001','SKU-002'])]
periods = [1, 2, 3]

# Costos (si no existen columnas, usar valores por defecto)
if {'production_cost','inventory_cost'}.issubset(set(df_products.columns)):
    prod_cost = df_products.set_index('sku')['production_cost'].to_dict()
    inv_cost = df_products.set_index('sku')['inventory_cost'].to_dict()
else:
    prod_cost = {p: 5.0 for p in products}
    inv_cost = {p: 0.5 for p in products}

# Demanda por periodo (si no hay df_demand, derivar simple de órdenes)
try:
    demand_dict = df_demand.set_index(['product','period'])['demand'].to_dict()
except Exception:
    demand_dict = {}
    for p in products:
        for t in periods:
            demand_dict[(p, t)] = int(df_orders[df_orders['sku'] == p]['qty'].sum() / len(periods)) if 'df_orders' in globals() else 50

# Capacidad máquina/almacén (valores por defecto si faltan)
MACHINE_CAPACITY = 480.0  # 8 horas * 60 min
STORAGE_CAPACITY = 10000.0

# Horas de máquina por unidad
if 'machine_hours' in df_products.columns:
    machine_hours = df_products.set_index('sku')['machine_hours'].to_dict()
else:
    # Convertir tiempos de proceso a horas si disponibles
    if 'proc_time_min' in df_products.columns:
        machine_hours = df_products.set_index('sku')['proc_time_min'].div(60).to_dict()
    else:
        machine_hours = {p: 0.1 for p in products}

# Variables de decisión
model = pulp.LpProblem('Production_Plan', pulp.LpMinimize)
production = pulp.LpVariable.dicts('Prod', ((p, t) for p in products for t in periods), lowBound=0)
inventory = pulp.LpVariable.dicts('Inv', ((p, t) for p in products for t in periods), lowBound=0)

# Función objetivo: minimizar costo total
model += (
    pulp.lpSum(
        prod_cost[p] * production[(p, t)] + inv_cost[p] * inventory[(p, t)]
        for p in products for t in periods
    ),
    'Total_Cost'
)
print('✅ Función objetivo definida: Minimizar (Costo Producción + Costo Inventario)')

✅ Función objetivo definida: Minimizar (Costo Producción + Costo Inventario)


## Próximos pasos
- Hipótesis y mejoras: introducir OR-Tools/PuLP para optimización exacta.
- Integración: exportar secuencias a MES/ERP; validar contra calendarios reales.

In [37]:
# Restricciones (ejecución segura con datos por defecto)
# Diccionario de demanda
try:
    demand_dict = df_demand.set_index(['product', 'period'])['demand'].to_dict()
except Exception:
    demand_dict = {}
    for p in products:
        for t in periods:
            demand_dict[(p, t)] = int(df_orders[df_orders['sku'] == p]['qty'].sum() / len(periods)) if 'df_orders' in globals() else 50

# Horas de máquina
try:
    machine_hours = df_products.set_index('sku')['machine_hours'].to_dict()
except Exception:
    if 'proc_time_min' in df_products.columns:
        machine_hours = df_products.set_index('sku')['proc_time_min'].div(60).to_dict()
    else:
        machine_hours = {p: 0.1 for p in products}

# 1. Balance de inventario
for p in products:
    for t in periods:
        if t == periods[0]:
            model += (
                inventory[(p, t)] == production[(p, t)] - demand_dict[(p, t)],
                f"Balance_{p}_t{t}"
            )
        else:
            model += (
                inventory[(p, t)] == inventory[(p, t-1)] + production[(p, t)] - demand_dict[(p, t)],
                f"Balance_{p}_t{t}"
            )

# 2. Capacidad de máquina
for t in periods:
    model += (
        pulp.lpSum(machine_hours.get(p, 0.1) * production[(p, t)] for p in products) <= MACHINE_CAPACITY,
        f"Machine_Capacity_t{t}"
    )

# 3. Capacidad de almacén
for t in periods:
    model += (
        pulp.lpSum(inventory[(p, t)] for p in products) <= STORAGE_CAPACITY,
        f"Storage_Capacity_t{t}"
    )

print("✅ Restricciones agregadas:")
print(f"   - Balance de inventario: {len(products) * len(periods)}")
print(f"   - Capacidad de máquina: {len(periods)}")
print(f"   - Capacidad de almacén: {len(periods)}")
print(f"\n📊 Total de restricciones: {len(model.constraints)}")

✅ Restricciones agregadas:
   - Balance de inventario: 30
   - Capacidad de máquina: 3
   - Capacidad de almacén: 3

📊 Total de restricciones: 36


## Datos realistas (synthetic)
Si faltan archivos en `data/raw/`, generamos datasets sintéticos con características realistas:
- Órdenes con fechas de entrega, prioridades, tamaños de lote variados.
- Productos con tiempos de proceso y setup dependientes de familia.
- Recursos con capacidades y calendarios típicos de planta.

## 7️⃣ Resolver Modelo

In [ ]:
# Generación de datasets sintéticos realistas (si faltan fuentes)
import numpy as np
import pandas as pd
from pathlib import Path

RAW_DIR = Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Productos: familias con distintos tiempos de proceso y setup
families = ['A','B','C']
skus = [f'SKU-{i:03d}' for i in range(1,11)]
prod_rows = []
for sku in skus:
    fam = np.random.choice(families, p=[0.4,0.4,0.2])
    proc = np.random.normal(loc={'A':5,'B':8,'C':12}[fam], scale=1.5)
    setup = np.random.normal(loc={'A':10,'B':15,'C':25}[fam], scale=3)
    prod_rows.append({'sku': sku, 'family': fam, 'proc_time_min': max(1, round(proc,1)), 'setup_time_min': max(1, round(setup,1))})
synthetic_products = pd.DataFrame(prod_rows)

# Recursos: dos máquinas con capacidad por hora
synthetic_resources = pd.DataFrame({
    'machine': ['M1','M2'],
    'capacity_units_per_hour': [600, 600],
    'calendar': ['08:00-17:00','08:00-17:00']
})

# Órdenes: mezcla de prioridades y due dates
dates = pd.date_range('2025-01-01','2025-01-07', freq='D')
order_rows = []
order_id = 1
for d in dates:
    for _ in range(np.random.randint(5,12)):
        sku = np.random.choice(skus)
        qty = int(np.random.choice([20,30,40,50,60,80,100], p=[0.1,0.15,0.2,0.2,0.15,0.1,0.1]))
        priority = int(np.random.choice([1,2,3], p=[0.2,0.5,0.3]))
        due = d + pd.Timedelta(days=np.random.choice([1,2,3,4], p=[0.3,0.4,0.2,0.1]))
        order_rows.append({'order_id': order_id, 'date': d, 'sku': sku, 'qty': qty, 'due_date': due, 'priority': priority})
        order_id += 1
synthetic_orders = pd.DataFrame(order_rows)

# Guardar si faltan archivos
for df, name in [(synthetic_products, 'products.csv'), (synthetic_resources, 'resources.csv'), (synthetic_orders, 'orders.csv')]:
    path = RAW_DIR / name
    if not path.exists():
        df.to_csv(path, index=False)
        print(f'🧪 Generado dataset sintético: {path}')
    else:
        print(f'ℹ️ Ya existe: {path}')

🧪 Generado dataset sintético: data\raw\products.csv
🧪 Generado dataset sintético: data\raw\resources.csv
🧪 Generado dataset sintético: data\raw\orders.csv


## 8️⃣ Extraer Resultados

In [ ]:
# Visualización: diagrama Gantt simple de la secuencia
import plotly.express as px
import plotly.io as pio
from pathlib import Path

if 'schedule_df' in globals() and not schedule_df.empty:
    gantt_df = schedule_df.copy()
    gantt_df['Task'] = 'M1 - ' + gantt_df['sku']
    fig_gantt = px.timeline(gantt_df, x_start='start', x_end='finish', y='Task', color='sku', title='Gantt de Secuencia (M1)')
    fig_gantt.update_yaxes(autorange='reversed')
    display(fig_gantt)
    out_dir = Path('data/processed/or08')
    out_dir.mkdir(parents=True, exist_ok=True)
    html_path = out_dir / 'gantt_M1.html'
    fig_gantt.write_html(html_path)
    print(f'✅ Exportado: {html_path}')
else:
    print('⚠️ No hay schedule_df para graficar.')

✅ Exportado: data\processed\or08\gantt_M1.html


## 9️⃣ Análisis de Resultados

In [38]:
# Resolver modelo y construir df_plan
model.solve(pulp.PULP_CBC_CMD(msg=0))
status = pulp.LpStatus[model.status]
objective = pulp.value(model.objective)
print(f"🧮 Solver status: {status} | Objetivo: {objective:.2f}")

rows = []
for p in products:
    for t in periods:
        rows.append({
            'product': p,
            'period': t,
            'production': production[(p, t)].varValue,
            'inventory': inventory[(p, t)].varValue,
        })
df_plan = pd.DataFrame(rows)
print('📄 Plan construido:')
display(df_plan.head())

🧮 Solver status: Optimal | Objetivo: 14640.00
📄 Plan construido:


,product,period,production,inventory
0,SKU-001,1,86.0,0.0
1,SKU-001,2,86.0,0.0
2,SKU-001,3,86.0,0.0
3,SKU-002,1,46.0,0.0
4,SKU-002,2,46.0,0.0


## 🔟 Validación de Restricciones

In [39]:
# Verificar uso de capacidad de máquina
machine_usage = []
for t in periods:
    total_hours = sum(
        machine_hours[p] * production[(p, t)].varValue
        for p in products
    )
    machine_usage.append({
        'period': t,
        'hours_used': total_hours,
        'capacity': MACHINE_CAPACITY,
        'utilization': total_hours / MACHINE_CAPACITY * 100
    })

df_machine = pd.DataFrame(machine_usage)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_machine['period'],
    y=df_machine['hours_used'],
    name='Horas Usadas',
    marker_color='lightblue'
))
fig.add_hline(
    y=MACHINE_CAPACITY,
    line_dash="dash",
    line_color="red",
    annotation_text="Capacidad Máxima"
)
fig.update_layout(
    title="Uso de Capacidad de Máquina",
    xaxis_title="Periodo",
    yaxis_title="Horas"
)
fig.show()

print("⚙️ Utilización de Máquina:")
display(df_machine)

# Verificar almacén
storage_usage = df_plan.groupby('period')['inventory'].sum().reset_index()
storage_usage['capacity'] = STORAGE_CAPACITY
storage_usage['utilization'] = storage_usage['inventory'] / STORAGE_CAPACITY * 100

print("\n📦 Utilización de Almacén:")
display(storage_usage)

⚙️ Utilización de Máquina:


,period,hours_used,capacity,utilization
0,1,92.438333,480.0,19.257986
1,2,92.438333,480.0,19.257986
2,3,92.438333,480.0,19.257986



📦 Utilización de Almacén:


,period,inventory,capacity,utilization
0,1,0.0,10000.0,0.0
1,2,0.0,10000.0,0.0
2,3,0.0,10000.0,0.0


## 1️⃣1️⃣ Desglose de Costos

In [ ]:
# Calcular costos por componente
df_plan['prod_cost'] = df_plan.apply(
    lambda row: prod_cost[row['product']] * row['production'], axis=1
)
df_plan['inv_cost'] = df_plan.apply(
    lambda row: inv_cost[row['product']] * row['inventory'], axis=1
)

total_prod_cost = df_plan['prod_cost'].sum()
total_inv_cost = df_plan['inv_cost'].sum()

# Pie chart de desglose
fig = go.Figure(data=[
    go.Pie(
        labels=['Costo Producción', 'Costo Inventario'],
        values=[total_prod_cost, total_inv_cost],
        hole=0.3
    )
])
fig.update_layout(title="Desglose de Costos Totales")
fig.show()

print("💰 Resumen de Costos:")
print(f"   - Producción: ${total_prod_cost:,.2f} ({total_prod_cost/(total_prod_cost+total_inv_cost)*100:.1f}%)")
print(f"   - Inventario: ${total_inv_cost:,.2f} ({total_inv_cost/(total_prod_cost+total_inv_cost)*100:.1f}%)")
print(f"   - TOTAL: ${total_prod_cost + total_inv_cost:,.2f}")

💰 Resumen de Costos:
   - Producción: $600.00 (100.0%)
   - Inventario: $0.00 (0.0%)
   - TOTAL: $600.00


## 🎓 Conclusiones

**Aprendizajes Clave:**
1. ✅ **Programación Lineal**: PuLP permite modelar problemas complejos de forma declarativa
2. ✅ **Trade-offs**: Modelo balancea costos de producción vs inventario automáticamente
3. ✅ **Capacidades**: Restricciones aseguran factibilidad operativa
4. ✅ **Solver CBC**: Resuelve modelos con decenas de variables en segundos

**Resultados de Negocio:**
- 💰 Costo óptimo: ~${total_prod_cost + total_inv_cost:,.0f}
- ⚙️ Utilización de máquina: {df_machine['utilization'].mean():.1f}% promedio
- 📦 Inventario promedio: {df_plan['inventory'].mean():.0f} unidades
- 🎯 100% de cumplimiento de demanda

**Decisiones Operativas:**
- Producir en periodos de baja demanda para aprovechar capacidad
- Mantener inventario buffer solo si costo de inventario < costo de overtime
- Productos con alto `machine_hours` priorizarse en periodos de baja demanda agregada

**Próximos Pasos:**
- Agregar costos de setup entre productos
- Incluir variables binarias para decisiones de producción (MIP)
- Multi-planta con costos de transporte (ver OR-09)
- Incertidumbre con programación estocástica

---

**🔗 Notebooks Relacionados:**
- [OR-01: Stock de Seguridad](../50_optimization_or/OR-01-stock_seguridad.ipynb)
- [OR-02: Políticas de Inventario](../50_optimization_or/OR-02-politicas_inventario.ipynb)
- [OR-09: Red Logística](../50_optimization_or/OR-09-network_optimization.ipynb)

## 🛠️ Funciones Reutilizables

In [ ]:
def solve_production_plan(
    products: list,
    periods: list,
    demand: dict,
    prod_cost: dict,
    inv_cost: dict,
    machine_hours: dict,
    machine_capacity: float,
    storage_capacity: float
) -> tuple:
    """
    Resuelve problema de production scheduling.
    
    Returns:
        (status, objective_value, production_vars, inventory_vars)
    """
    model = pulp.LpProblem("Production_Plan", pulp.LpMinimize)
    
    # Variables
    X = pulp.LpVariable.dicts("Prod", ((p, t) for p in products for t in periods), lowBound=0)
    I = pulp.LpVariable.dicts("Inv", ((p, t) for p in products for t in periods), lowBound=0)
    
    # Objetivo
    model += pulp.lpSum(
        prod_cost[p] * X[(p, t)] + inv_cost[p] * I[(p, t)]
        for p in products for t in periods
    )
    
    # Restricciones
    for p in products:
        for t in periods:
            if t == periods[0]:
                model += I[(p, t)] == X[(p, t)] - demand[(p, t)]
            else:
                model += I[(p, t)] == I[(p, t-1)] + X[(p, t)] - demand[(p, t)]
    
    for t in periods:
        model += pulp.lpSum(machine_hours[p] * X[(p, t)] for p in products) <= machine_capacity
        model += pulp.lpSum(I[(p, t)] for p in products) <= storage_capacity
    
    # Resolver
    model.solve(pulp.PULP_CBC_CMD(msg=0))
    
    return pulp.LpStatus[model.status], pulp.value(model.objective), X, I

# Ejemplo de uso:
# status, cost, prod, inv = solve_production_plan(products, periods, demand_dict, ...)

## 🗓️ Buenas Prácticas de Calendarización
Para que las secuencias y planes sean aplicables en planta, incorpora reglas de calendario y mantenimiento:
- Ventanas operativas por máquina: define turnos y pausas (e.g., 08:00–17:00, pausa 12:00–13:00).
- Mantenimientos preventivos: bloquea periodos (e.g., M1 mantenimiento semanal 2h los viernes 10:00–12:00).
- Cambios de turno: evita iniciar setups largos cerca del fin de turno.
- Festivos y paros: integra calendario corporativo para evitar asignaciones inviables.
- Buffers: añade holguras antes/después de trabajos críticos para absorber variabilidad.

Ejemplo simple de ventanas por máquina (estructura sugerida):
```python
calendarios = {
    'M1': [
        {'dia': 'L-V', 'inicio': '08:00', 'fin': '17:00'},
        {'dia': 'V', 'inicio': '10:00', 'fin': '12:00', 'tipo': 'mantenimiento'}
    ],
    'M2': [
        {'dia': 'L-V', 'inicio': '08:00', 'fin': '17:00'}
    ]
}
# En el modelo/heurística: no programar tareas fuera de ventanas y respetar bloqueos.
```
Sugerencias para el modelo:
- Añadir restricciones de “no solape” con ventanas activas.
- Penalizar setups que crucen fin de turno.
- Incluir variables binarias de activación por slot horario (ver bloque OR-Tools).

## Variante Multi-Máquina (M1/M2)
Asignación simple por carga: distribuir órdenes entre M1 y M2 balanceando tiempo de proceso acumulado y considerando setups.

In [40]:
# Asignación y secuenciación multi-máquina (heurística balanceo de carga)
import pandas as pd
from collections import defaultdict
 
# Preparar jobs con tiempos
jobs_mm = df_orders.merge(df_products[['sku','proc_time_min','setup_time_min']], on='sku', how='left').copy()
jobs_mm['proc_time_min'] = jobs_mm['proc_time_min'].fillna(5)
jobs_mm['setup_time_min'] = jobs_mm['setup_time_min'].fillna(10)
jobs_mm = jobs_mm.sort_values(['due_date','priority','proc_time_min']).reset_index(drop=True)
 
machines = ['M1','M2']
machine_load = {m: pd.Timestamp(jobs_mm['date'].min()) for m in machines}
machine_last_sku = {m: None for m in machines}
schedules = {m: [] for m in machines}
 
# Asignar cada job a la máquina con menor tiempo de finalización estimado
for _, r in jobs_mm.iterrows():
    best_m = None
    best_finish = None
    for m in machines:
        setup = r['setup_time_min'] if (machine_last_sku[m] is not None and machine_last_sku[m] != r['sku']) else 0
        start = machine_load[m] + pd.Timedelta(minutes=setup)
        finish = start + pd.Timedelta(minutes=r['proc_time_min']*r['qty'])
        if best_finish is None or finish < best_finish:
            best_finish = finish
            best_m = m
    # Asignar al mejor
    m = best_m
    setup = r['setup_time_min'] if (machine_last_sku[m] is not None and machine_last_sku[m] != r['sku']) else 0
    start = machine_load[m] + pd.Timedelta(minutes=setup)
    finish = start + pd.Timedelta(minutes=r['proc_time_min']*r['qty'])
    tardiness = max(pd.Timedelta(0), finish - r['due_date'])
    schedules[m].append({
        'machine': m, 'order_id': r['order_id'], 'sku': r['sku'], 'start': start, 'finish': finish,
        'setup_min': setup, 'proc_min': r['proc_time_min']*r['qty'], 'tardiness_min': tardiness.total_seconds()/60.0
    })
    machine_load[m] = finish
    machine_last_sku[m] = r['sku']
 
# Consolidar y KPIs
schedule_mm_df = pd.concat([pd.DataFrame(v) for v in schedules.values()], ignore_index=True)
print('📅 Secuencias multi-máquina:')
display(schedule_mm_df.head())
 
kpis_mm = schedule_mm_df.groupby('machine').agg(orders=('order_id','count'),
                                              total_setup_min=('setup_min','sum'),
                                              avg_tardiness_min=('tardiness_min','mean')).reset_index()
print('📈 KPIs por máquina:')
display(kpis_mm)
 
# Exportar artefactos
from pathlib import Path
OUT_MM = Path('data/processed/or08')
OUT_MM.mkdir(parents=True, exist_ok=True)
schedule_mm_path = OUT_MM / 'schedule_multi_machine.csv'
schedule_mm_df.to_csv(schedule_mm_path, index=False)
print(f'✅ Exportado: {schedule_mm_path}')
 
# Gantt por máquina
import plotly.express as px
fig_m1 = px.timeline(schedule_mm_df[schedule_mm_df['machine']=='M1'].assign(Task=lambda d: 'M1 - ' + d['sku']),
                     x_start='start', x_end='finish', y='Task', color='sku', title='Gantt M1')
fig_m1.update_yaxes(autorange='reversed')
fig_m2 = px.timeline(schedule_mm_df[schedule_mm_df['machine']=='M2'].assign(Task=lambda d: 'M2 - ' + d['sku']),
                     x_start='start', x_end='finish', y='Task', color='sku', title='Gantt M2')
fig_m2.update_yaxes(autorange='reversed')
display(fig_m1)
display(fig_m2)
fig_m1.write_html(OUT_MM / 'gantt_M1_multi.html')
fig_m2.write_html(OUT_MM / 'gantt_M2_multi.html')
print('✅ Exportados: gantt_M1_multi.html, gantt_M2_multi.html')

📅 Secuencias multi-máquina:


,machine,order_id,sku,start,finish,setup_min,proc_min,tardiness_min
0,M1,7,SKU-010,2025-01-01 00:00:00.000000000,2025-01-01 06:49:59.999999999,0.0,410.0,0.0
1,M1,1,SKU-008,2025-01-01 07:05:17.999999999,2025-01-01 09:29:17.999999999,15.3,144.0,0.0
2,M1,15,SKU-010,2025-01-01 09:44:53.999999999,2025-01-01 11:47:53.999999998,15.6,123.0,0.0
3,M1,22,SKU-009,2025-01-01 12:05:41.999999998,2025-01-01 16:49:41.999999998,17.8,284.0,0.0
4,M1,17,SKU-007,2025-01-01 17:06:29.999999998,2025-01-01 23:01:29.999999998,16.8,355.0,0.0


📈 KPIs por máquina:


,machine,orders,total_setup_min,avg_tardiness_min
0,M1,30,329.6,0.0
1,M2,30,333.2,0.0


✅ Exportado: data\processed\or08\schedule_multi_machine.csv


✅ Exportados: gantt_M1_multi.html, gantt_M2_multi.html


## Comparativa KPIs: Mono vs Multi-Máquina
Tabla y comentarios rápidos sobre setups, tardanza y órdenes atendidas.

## Interpretación de KPIs y Recomendaciones
- **`setup_min`**: minutos destinados a cambios de preparación; valores altos indican secuencias con muchos cambios de SKU o familias. Reducir cambiando reglas (agrupar por SKU/familia).
- **`avg_tardiness_min`**: tardanza promedio respecto a `due_date`; presión de servicio. Si es alto, considerar más capacidad, priorización por fechas, o dividir lotes críticos.
- **`orders`**: volumen atendido; comparar entre variantes para capacidad efectiva.
- **Mono vs Multi**: multi suele reducir `setup_min` al paralelizar y mejorar tiempos de ciclo, pero puede requerir coordinación de calendarios y setups entre máquinas.
Recomendaciones prácticas:
- Agrupar trabajos por familia para minimizar setups.
- Reservar ventanas de capacidad para órdenes de alta prioridad/fecha cercana.
- Ajustar `proc_time_min` y `setup_time_min` con datos reales de MES/ERP.
- Revisar restricciones de calendario por máquina (turnos, mantenimiento) antes de desplegar.

In [41]:
# Construir comparativa KPIs mono vs multi-máquina
import pandas as pd
from pathlib import Path
 
mono = pd.DataFrame([{
    'setup_min': kpis_native.get('total_setup_min', kpis.get('total_setup_min', 0)),
    'avg_tardiness_min': kpis_native.get('avg_tardiness_min', kpis.get('avg_tardiness_min', 0)),
    'orders': kpis_native.get('orders', kpis.get('orders', 0)),
    'variant': 'mono'
}])
multi = kpis_mm.copy()
multi = multi.rename(columns={'total_setup_min':'setup_min','avg_tardiness_min':'avg_tardiness_min','orders':'orders'})
multi['variant'] = 'multi'
 
comp = pd.concat([mono, multi], ignore_index=True)
print('📊 Comparativa KPIs:')
display(comp)
 
out_dir = Path('data/processed/or08')
out_dir.mkdir(parents=True, exist_ok=True)
comp.to_csv(out_dir / 'kpi_comparison_mono_vs_multi.csv', index=False)
print('✅ Exportado: kpi_comparison_mono_vs_multi.csv')

📊 Comparativa KPIs:


,setup_min,avg_tardiness_min,orders,variant,machine
0,581.5,1254.331667,60,mono,NaN
1,329.6,0.000000,30,multi,M1
2,333.2,0.000000,30,multi,M2


✅ Exportado: kpi_comparison_mono_vs_multi.csv


## OR-Tools (MIP básico, opcional)
Ejemplo compacto de asignación por periodos con restricciones de capacidad utilizando OR-Tools CP-SAT; adecuado para problemas con decisiones binarias.

In [31]:
# OR-Tools CP-SAT: asignación binaria simple por periodos (demo)
try:
    from ortools.sat.python import cp_model
    import numpy as np
    import pandas as pd
    
    model = cp_model.CpModel()
    products_rt = products
    periods_rt = periods
    # Variables binarias: producir (1) o no (0) por producto y periodo
    X = {}
    for p in products_rt:
        for t in periods_rt:
            X[(p,t)] = model.NewBoolVar(f'produce_{p}_{t}')
    
    # Demandas mínimas: producir en al menos un periodo si demanda > 0
    for p in products_rt:
        demand_total = sum(demand_dict[(p,t)] for t in periods_rt)
        if demand_total > 0:
            model.Add(sum(X[(p,t)] for t in periods_rt) >= 1)
    
    # Capacidad por periodo: limitar número de productos activos (proxy de capacidad)
    max_active = min(len(products_rt), 3)
    for t in periods_rt:
        model.Add(sum(X[(p,t)] for p in products_rt) <= max_active)
    
    # Objetivo: minimizar tardanza proxy (preferir periodos tempranos)
    # Penalizar activaciones en periodos tardíos
    weights = {t: t for t in periods_rt}
    model.Minimize(sum(weights[t] * X[(p,t)] for p in products_rt for t in periods_rt))
    
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 5.0
    res = solver.Solve(model)
    print('🧮 OR-Tools status:', res)
    rows = []
    for p in products_rt:
        for t in periods_rt:
            rows.append({'product': p, 'period': t, 'active': solver.Value(X[(p,t)])})
    df_rt = pd.DataFrame(rows)
    print('📄 Plan OR-Tools (activo por periodo):')
    display(df_rt.pivot(index='period', columns='product', values='active'))
except Exception as e:
    print('⚠️ OR-Tools no disponible o error en demo:', e)

🧮 OR-Tools status: 4
📄 Plan OR-Tools (activo por periodo):


product,SKU-001,SKU-002
period,,
1,1,1
2,0,0
3,0,0


In [45]:
# Validador de secuencias contra ventanas de calendario por máquina
# - Marca tareas fuera de turno
# - Ajusta (opcionales) inicios para respetar ventanas
# - Exporta reporte de violaciones y artefactos ajustados
import pandas as pd
from pathlib import Path

# Ejemplo de calendarios (puedes reemplazar por fuentes reales)
calendarios = {
    'M1': [
        {'dias': 'L-V', 'inicio': '08:00', 'fin': '17:00'},
        {'dias': 'V',   'inicio': '10:00', 'fin': '12:00', 'tipo': 'mantenimiento'}
    ],
    'M2': [
        {'dias': 'L-V', 'inicio': '08:00', 'fin': '17:00'}
    ]
}

# Helper: día a letra (L,M,X,J,V,S,D)
_dow_map = {0:'L',1:'M',2:'X',3:'J',4:'V',5:'S',6:'D'}

orden_dias = ['L','M','X','J','V','S','D']

def ventanas_activas_para(fecha: pd.Timestamp, reglas: list) -> list:
    dia = _dow_map[fecha.weekday()]
    activas = []
    for r in reglas:
        dias = r.get('dias','L-V')
        # Rango L-V, o día específico
        if '-' in dias:
            ini_d, fin_d = dias.split('-')
            if orden_dias.index(ini_d) <= orden_dias.index(dia) <= orden_dias.index(fin_d):
                activas.append(r)
        else:
            if dias == dia:
                activas.append(r)
    return activas

# Construir ventanas (sin mantenimiento) para una fecha concreta

def construir_ventanas_del_dia(fecha: pd.Timestamp, reglas: list):
    activas = ventanas_activas_para(fecha, reglas)
    ventanas = []
    for v in activas:
        if v.get('tipo') == 'mantenimiento':
            continue
        v_start = pd.Timestamp(fecha.date().strftime('%Y-%m-%d') + ' ' + v['inicio'])
        v_end   = pd.Timestamp(fecha.date().strftime('%Y-%m-%d') + ' ' + v['fin'])
        ventanas.append((v_start, v_end))
    # ordenar por inicio
    ventanas.sort(key=lambda t: t[0])
    return ventanas

# Buscar siguiente ventana disponible (mismo día luego próximos días)

def buscar_siguiente_slot(start: pd.Timestamp, dur: pd.Timedelta, reglas: list) -> tuple:
    # intentar en el mismo día
    ventanas = construir_ventanas_del_dia(start, reglas)
    for vs, ve in ventanas:
        s = max(start, vs)
        f = s + dur
        if f <= ve:
            return s, f
    # si no cabe en el mismo día, mover a siguiente día laboral con primera ventana
    for i in range(1, 8):  # buscar como máximo la próxima semana
        cand = start + pd.Timedelta(days=i)
        ventanas2 = construir_ventanas_del_dia(cand, reglas)
        if not ventanas2:
            continue
        vs, ve = ventanas2[0]
        s = vs
        f = s + dur
        if f <= ve:
            return s, f
        # si tampoco cabe, recortar al final de ventana (opción conservadora)
        return s, ve
    # si no hay ventanas, devolver original
    return start, start + dur

# Valida un dataframe de schedule con columnas: machine, start, finish

def validar_schedule(df_sched: pd.DataFrame, calendarios: dict, ajustar: bool=False) -> pd.DataFrame:
    rows = []
    for _, r in df_sched.iterrows():
        m = r.get('machine','M1')
        start = pd.to_datetime(r['start'])
        finish = pd.to_datetime(r['finish'])
        reglas = calendarios.get(m, [])
        activas = ventanas_activas_para(start, reglas)
        # Construir ventanas del día (excluyendo mantenimiento como ventana activa)
        ventanas = construir_ventanas_del_dia(start, reglas)
        # Evaluar cumplimiento
        cumple = False
        solape_mant = False
        for v in activas:
            if v.get('tipo') == 'mantenimiento':
                mant_start = pd.Timestamp(start.date().strftime('%Y-%m-%d') + ' ' + v['inicio'])
                mant_end   = pd.Timestamp(start.date().strftime('%Y-%m-%d') + ' ' + v['fin'])
                if not (finish <= mant_start or start >= mant_end):
                    solape_mant = True
        for vs, ve in ventanas:
            if start >= vs and finish <= ve:
                cumple = True
                break
        start_adj, finish_adj = start, finish
        if ajustar:
            dur = finish - start
            start_adj, finish_adj = buscar_siguiente_slot(start, dur, reglas)
        rows.append({
            'machine': m,
            'order_id': r.get('order_id'),
            'sku': r.get('sku'),
            'start': start,
            'finish': finish,
            'start_adj': start_adj,
            'finish_adj': finish_adj,
            'within_window': cumple,
            'overlaps_maintenance': solape_mant
        })
    return pd.DataFrame(rows)

# Seleccionar schedule a validar: multi-máquina si existe, si no mono
if 'schedule_mm_df' in globals() and not schedule_mm_df.empty:
    df_sched_in = schedule_mm_df.copy()
elif 'schedule_df' in globals() and not schedule_df.empty:
    df_sched_in = schedule_df.copy()
    df_sched_in['machine'] = 'M1'
else:
    df_sched_in = pd.DataFrame()

if df_sched_in.empty:
    print('⚠️ No hay secuencia para validar contra calendario.')
else:
    # Validación sin ajuste (reporte de violaciones)
    reporte = validar_schedule(df_sched_in, calendarios, ajustar=False)
    print('🧪 Validación de calendario (primeras filas):')
    display(reporte.head())
    violaciones = reporte[(~reporte['within_window']) | (reporte['overlaps_maintenance'])]
    print(f"❗ Violaciones detectadas: {len(violaciones)}")
    display(violaciones.head())

    # Validación con ajuste y KPIs recalculados (tardanza vs due_date si está disponible)
    reporte_adj = validar_schedule(df_sched_in, calendarios, ajustar=True)
    # Recalcular tardanza con tiempos ajustados
    if 'due_date' in df_orders.columns:
        df_due = df_orders[['order_id','due_date']].copy()
        rep_kpi = reporte_adj.merge(df_due, on='order_id', how='left')
        rep_kpi['due_date'] = pd.to_datetime(rep_kpi['due_date'], errors='coerce')
        rep_kpi['tardiness_min_adj'] = rep_kpi.apply(
            lambda r: max(pd.Timedelta(0), pd.to_datetime(r['finish_adj']) - (r['due_date'] if pd.notnull(r['due_date']) else pd.to_datetime(r['finish_adj']))).total_seconds()/60.0,
            axis=1
        )
    else:
        rep_kpi = reporte_adj.copy()
        rep_kpi['tardiness_min_adj'] = 0.0

    kpi_adjusted = {
        'orders': int(len(rep_kpi)),
        'violations': int(len(violaciones)),
        'avg_tardiness_min_adj': float(rep_kpi['tardiness_min_adj'].mean()),
    }
    print('📈 KPIs ajustados: ', kpi_adjusted)

    # Exportes
    out_dir = Path('data/processed/or08')
    out_dir.mkdir(parents=True, exist_ok=True)
    path_rep = out_dir / 'schedule_calendar_validation.csv'
    reporte.to_csv(path_rep, index=False)
    print(f'✅ Exportado: {path_rep}')
    path_adj = out_dir / 'schedule_adjusted.csv'
    reporte_adj.to_csv(path_adj, index=False)
    print(f'✅ Exportado: {path_adj}')
    import json
    with open(out_dir / 'kpi_adjusted.json', 'w', encoding='utf-8') as f:
        json.dump(kpi_adjusted, f, ensure_ascii=False, indent=2)
    print(f"✅ Exportado: {out_dir / 'kpi_adjusted.json'}")


🧪 Validación de calendario (primeras filas):


,machine,order_id,sku,start,finish,start_adj,finish_adj,within_window,overlaps_maintenance
0,M1,7,SKU-010,2025-01-01 00:00:00.000000000,2025-01-01 06:49:59.999999999,2025-01-01 00:00:00.000000000,2025-01-01 06:49:59.999999999,False,False
1,M1,1,SKU-008,2025-01-01 07:05:17.999999999,2025-01-01 09:29:17.999999999,2025-01-01 07:05:17.999999999,2025-01-01 09:29:17.999999999,False,False
2,M1,15,SKU-010,2025-01-01 09:44:53.999999999,2025-01-01 11:47:53.999999998,2025-01-01 09:44:53.999999999,2025-01-01 11:47:53.999999998,True,False
3,M1,22,SKU-009,2025-01-01 12:05:41.999999998,2025-01-01 16:49:41.999999998,2025-01-01 12:05:41.999999998,2025-01-01 16:49:41.999999998,True,False
4,M1,17,SKU-007,2025-01-01 17:06:29.999999998,2025-01-01 23:01:29.999999998,2025-01-01 17:06:29.999999998,2025-01-01 23:01:29.999999998,False,False


❗ Violaciones detectadas: 50


,machine,order_id,sku,start,finish,start_adj,finish_adj,within_window,overlaps_maintenance
0,M1,7,SKU-010,2025-01-01 00:00:00.000000000,2025-01-01 06:49:59.999999999,2025-01-01 00:00:00.000000000,2025-01-01 06:49:59.999999999,False,False
1,M1,1,SKU-008,2025-01-01 07:05:17.999999999,2025-01-01 09:29:17.999999999,2025-01-01 07:05:17.999999999,2025-01-01 09:29:17.999999999,False,False
4,M1,17,SKU-007,2025-01-01 17:06:29.999999998,2025-01-01 23:01:29.999999998,2025-01-01 17:06:29.999999998,2025-01-01 23:01:29.999999998,False,False
5,M1,20,SKU-009,2025-01-01 23:19:17.999999998,2025-01-02 02:52:17.999999998,2025-01-01 23:19:17.999999998,2025-01-02 02:52:17.999999998,False,False
6,M1,3,SKU-004,2025-01-02 02:56:35.999999998,2025-01-02 05:38:35.999999998,2025-01-02 02:56:35.999999998,2025-01-02 05:38:35.999999998,False,False


📈 KPIs ajustados:  {'orders': 60, 'violations': 50, 'avg_tardiness_min_adj': 64.86666666666666}
✅ Exportado: data\processed\or08\schedule_calendar_validation.csv
✅ Exportado: data\processed\or08\schedule_adjusted.csv
✅ Exportado: data\processed\or08\kpi_adjusted.json


In [44]:
# Visualización Gantt ajustada y comparativa de KPIs (antes vs después)
import pandas as pd
import plotly.express as px
from pathlib import Path

out_dir = Path('data/processed/or08')
out_dir.mkdir(parents=True, exist_ok=True)

# Construir Gantt ajustado por máquina, si se dispone de reporte ajustado
try:
    rep_adj_path = out_dir / 'schedule_adjusted.csv'
    reporte_adj = pd.read_csv(rep_adj_path, parse_dates=['start','finish','start_adj','finish_adj'])
    if 'machine' not in reporte_adj.columns:
        reporte_adj['machine'] = 'M1'
    # Gantt por máquina con tiempos ajustados
    for m in sorted(reporte_adj['machine'].unique()):
        dfm = reporte_adj[reporte_adj['machine'] == m].copy()
        if dfm.empty:
            continue
        dfm['Task'] = f'{m} - ' + dfm['sku'].astype(str)
        fig_adj = px.timeline(dfm, x_start='start_adj', x_end='finish_adj', y='Task', color='sku', title=f'Gantt Ajustado {m}')
        fig_adj.update_yaxes(autorange='reversed')
        fig_adj.write_html(out_dir / f'gantt_{m}_adjusted.html')
        print(f"✅ Exportado: {out_dir / f'gantt_{m}_adjusted.html'}")
except Exception as e:
    print('⚠️ No se pudo generar Gantt ajustado:', e)

# Comparativa KPIs antes vs después del ajuste
try:
    # KPIs originales
    kpis_orig = {
        'variant': 'antes',
        'orders': int(kpis.get('orders', 0)),
        'setup_min': float(kpis.get('total_setup_min', 0)),
        'avg_tardiness_min': float(kpis.get('avg_tardiness_min', 0)),
    }
    # KPIs ajustados
    import json
    with open(out_dir / 'kpi_adjusted.json', 'r', encoding='utf-8') as f:
        kpi_adjusted = json.load(f)
    kpis_after = {
        'variant': 'despues',
        'orders': int(kpi_adjusted.get('orders', 0)),
        'setup_min': None,  # no se recalcula setup en el ajuste simple
        'avg_tardiness_min': float(kpi_adjusted.get('avg_tardiness_min_adj', 0.0)),
        'violations': int(kpi_adjusted.get('violations', 0)),
    }
    comp_adj = pd.DataFrame([kpis_orig, kpis_after])
    display(comp_adj)
    comp_adj.to_csv(out_dir / 'kpi_comparison_adjusted.csv', index=False)
    print(f"✅ Exportado: {out_dir / 'kpi_comparison_adjusted.csv'}")
except Exception as e:
    print('⚠️ No se pudo construir comparativa ajustada:', e)


✅ Exportado: data\processed\or08\gantt_M1_adjusted.html
✅ Exportado: data\processed\or08\gantt_M2_adjusted.html


,variant,orders,setup_min,avg_tardiness_min,violations
0,antes,60,581.5,1254.331667,NaN
1,despues,60,NaN,0.000000,50.0


✅ Exportado: data\processed\or08\kpi_comparison_adjusted.csv


## Interpretación de la comparativa “Antes vs Después”
- **Violations**: número de tareas fuera de turno o en mantenimiento antes del ajuste; alto indica secuencias poco operables.
- **Avg tardiness (después)**: tardanza recalculada con tiempos ajustados a ventanas; si baja, el ajuste mejoró cumplimiento de fechas.
- **Setup**: no se recalcula en este ajuste simple; para casos reales, conviene replanificar setups largos para no cruzar fin de turno.

Buenas prácticas para mejorar el resultado:
- Buscar la “siguiente” ventana disponible si el slot actual no cubre toda la duración (en vez de recortar).
- Reordenar trabajos para que setups largos caigan dentro de ventanas amplias.
- Introducir costos/penalizaciones por cruzar ventanas y binarios de activación por slot (ver bloque OR-Tools).
- Validar contra calendario corporativo (festivos, paros, mantenimiento) y ajustar buffers.


<div style="width: 100%; clear: both; margin: 0 0 20px 0; border-top: 1px solid #eaecef; padding-top: 24px;"><div style="display: flex; justify-content: space-between; align-items: center; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif;"><div style="flex: 1; text-align: left;"><a href="OR-07-safety_stock_intro.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">← Anterior: [OR-07-safety_stock_intro.ipynb](../50_optimization_or/OR-07-safety_stock_intro.ipynb)</a></div><div style="flex: 1; text-align: center; font-size: 14px;"><a href="../../README.md" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📑 Índice</a><span style="color: #6a737d;">|</span><a href="../../config/notebooks_index.yml" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📋 Catálogo</a></div><div style="flex: 1; text-align: right;"><a href="OR-09-network_optimization.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">Siguiente: OR-09 →</a></div></div></div>